# improve

In [ ]:
#| default_exp improve

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

A round improves Ramabana's learned folder from one finished session, in the manner of Harness²'s Propose–Probe–Compose step. It replays the session's request `k` times from the commit the session started at, under the current learned folder. An editor model reads those runs and edits a copy of the folder. The request is replayed once more under the copy. The copy is kept only when that run delivered a reply and took a cheaper route than every base run. Kept copies are committed in the learned folder's own git repository, one commit per round.

In [ ]:
#| export
import json, re, shutil, subprocess
from collections import Counter
from datetime import datetime, timezone
from fastcore.docments import frontmatter
from fastcore.xtras import Path

from dhrona.core import read_history, assess_turn, tool_args

RAMABANA = 'ramabana'   #: the command a round runs; another install replays with that one
LEARNED_MAX = 2000      #: characters of always-on `AGENTS.md` a candidate may carry
_HIDDEN = shutil.ignore_patterns('.*')   # `.git`, and the `.kosha` index Ramabana writes into every folder it opens

## Runs

`replay_clone` checks out the commit a session started from, in a throwaway clone. `run_turn` runs one headless Ramabana turn in a fresh config folder, so no history, extension, skill or project note from the person's own config takes part. It reads the turn back from that folder's history, which records the request, every tool call and the reply. Writes are refused unless `approve` allows them.

In [ ]:
#| export
def replay_clone(start, dest):
    "A throwaway clone of `start['repo']` checked out at `start['head']`, in `dest`."
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '-q', '--no-hardlinks', start['repo'], str(dest)], check=True)   # a replay's edits never reach the source's objects
    subprocess.run(['git', '-C', str(dest), 'checkout', '-q', '--detach', start['head']], check=True)
    return dest

def run_turn(
    prompt,          # the request
    roots,           # folders the turn may work in
    cfg,             # a fresh config folder for this one turn
    model=None,      # Ramabana's routing default when None
    learned=None,    # a learned folder to copy in as `<cfg>/learned`
    approve='ask',   # `ask` refuses every write in a headless turn; `edits` allows file edits
    exe=RAMABANA,
    timeout=1800,    # seconds before the run is abandoned
):
    "One headless Ramabana turn; returns its history row: `prompt`, `activity`, `reply`, `usage`."
    cfg = Path(cfg)
    cfg.mkdir(parents=True, exist_ok=True)
    if learned is not None: shutil.copytree(learned, cfg/'learned', ignore=_HIDDEN)
    cmd = [exe, '--root', ','.join(map(str, roots)), '--cfg', str(cfg), '--approve', approve,
           '--no-web', '--tmux', 'off', '--pane', 'off', *(['--model', model] if model else []), prompt]
    try: p = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: raise RuntimeError(f'the run in {cfg} was still going after {timeout}s')
    (cfg/'run.log').write_text(f'exit {p.returncode}\n\n{p.stderr}')
    hist = cfg/'agent-history.jsonl'
    if not hist.exists() or not (turns := read_history(hist)): raise RuntimeError(f'no turn was recorded; see {cfg/"run.log"}')
    return turns[-1]

## Comparing runs

A route costs its route findings first, then its failed or refused calls, then its calls. `better` keeps a candidate only when it delivered a reply and costs less than every base run. A candidate that merely ties the best base run is not kept.

In [ ]:
#| export
def route_cost(turn):
    "`(route findings, failed or refused calls, calls)` for one turn; a smaller tuple is a cheaper route."
    a = assess_turn(turn)
    return (len(a.findings), sum(x.get('ok') is False for x in turn.get('activity') or []), a.calls)

def delivered(turn):
    "Whether the turn ended with a reply and no error."
    return bool((turn.get('reply') or '').strip()) and not turn.get('error')

def better(cand, bases):
    "Whether `cand` delivered a reply and costs less than every turn in `bases`."
    return delivered(cand) and all(route_cost(cand) < route_cost(b) for b in bases)

In [ ]:
def turn(*acts, reply='done'): return {'prompt': 'sizes of the big files', 'reply': reply, 'activity': [dict(tool=t, ok=ok, args=a) for t, ok, a in acts]}
ls, ls2 = ('ls', True, {'path': ''}), ('ls', True, {'path': 'src'})
refused = ('run_shell', False, {'command': 'stat -f %z *.py'})
shell_grep = ('run_shell', True, {'command': 'grep -rn def .'})
base = turn(ls, refused, ls2)
test_eq(route_cost(base), (0, 1, 3))
test_eq(route_cost(turn(shell_grep)), (1, 0, 1))     # a shell grep is a bypass finding
assert better(turn(ls), [base, turn(ls, ls2)])
assert not better(turn(ls, ls2), [base, turn(ls, ls2)]) # ties the best base run
assert not better(turn(ls, reply=''), [base])

## The learned folder

A candidate may hold `AGENTS.md`, up to `LEARNED_MAX` characters, and skills at `skills/<name>/SKILL.md` whose frontmatter names the skill and describes it. `check_learned` lists anything else as a problem, and a round runs no candidate that has one. Hidden entries, such as `.git` and the `.kosha` index Ramabana writes into a folder it opens, are not part of the folder. `keep` replaces the learned folder's files with the candidate's and commits them in the folder's own repository, authored by Dhrona. The first time, it commits what the folder already held before replacing it. It refuses a folder with uncommitted changes. `git revert` undoes one round.

In [ ]:
#| export
def _files(folder):
    "Every file under `folder` outside hidden folders, as relative path to text."
    folder = Path(folder)
    return {p.relative_to(folder).as_posix(): p.read_text(errors='replace') for p in sorted(folder.rglob('*'))
            if p.is_file() and not any(s.startswith('.') for s in p.relative_to(folder).parts)}

def check_learned(folder):
    "Problems that rule out a candidate learned folder; empty when it may be tried."
    out = []
    for rel, text in _files(folder).items():
        if rel == 'AGENTS.md':
            if len(text) > LEARNED_MAX: out.append(f'AGENTS.md is over {LEARNED_MAX} characters')
        elif re.fullmatch(r'skills/[a-z0-9_-]+/SKILL\.md', rel):
            name, meta = rel.split('/')[1], frontmatter(text)[0]
            if meta.get('name') != name or not meta.get('description'):
                out.append(f'{rel} needs `name: {name}` and a `description` in its frontmatter')
        else: out.append(f'{rel} is neither AGENTS.md nor skills/<name>/SKILL.md')
    return out

def keep(candidate, learned, message):
    "Replace `learned`'s files with `candidate`'s and commit them in `learned`'s own repository; returns the short commit."
    learned = Path(learned)
    learned.mkdir(parents=True, exist_ok=True)
    def git(*a): return subprocess.run(['git', '-C', str(learned), *a], check=True, capture_output=True, text=True).stdout.strip()
    def commit(msg): git('add', '-A'); git('-c', 'user.name=Dhrona', '-c', 'user.email=dhrona@localhost', 'commit', '-q', '-m', msg)
    if not (learned/'.git').exists():
        git('init', '-q')
        if _files(learned): commit('The learned folder before Dhrona kept a round')
    elif git('status', '--porcelain'): raise RuntimeError(f'{learned} has uncommitted changes; commit or discard them before a round keeps a candidate')
    for p in learned.iterdir():
        if not p.name.startswith('.'): shutil.rmtree(p) if p.is_dir() else p.unlink()
    shutil.copytree(candidate, learned, dirs_exist_ok=True, ignore=_HIDDEN)
    commit(message)
    return git('rev-parse', '--short', 'HEAD')

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    d = Path(d); cand, learned = d/'cand', d/'learned'
    (cand/'skills'/'sizes').mkdir(parents=True)
    (cand/'AGENTS.md').write_text('Read file sizes from `ls`; it reports them.\n')
    (cand/'skills'/'sizes'/'SKILL.md').write_text('---\nname: sizes\ndescription: Compare file sizes.\n---\nUse ls.\n')
    (cand/'.kosha').mkdir(); (cand/'.kosha'/'code.db').write_bytes(b'\x95\x00')   # the index Ramabana writes into a folder it opens
    test_eq(check_learned(cand), [])
    (cand/'notes.txt').write_text('x')
    (cand/'skills'/'bad').mkdir(); (cand/'skills'/'bad'/'SKILL.md').write_text('no frontmatter')
    test_eq(len(check_learned(cand)), 2)
    (cand/'notes.txt').unlink(); shutil.rmtree(cand/'skills'/'bad')
    (learned/'skills'/'stale').mkdir(parents=True); (learned/'skills'/'stale'/'SKILL.md').write_text('old')
    keep(cand, learned, 'Read sizes from ls')
    assert 'skills/stale/SKILL.md' in subprocess.run(['git', '-C', str(learned), 'show', '--stat', 'HEAD~1'], capture_output=True, text=True).stdout   # what was there before Dhrona is kept in history
    (learned/'AGENTS.md').write_text('a hand edit')
    test_fail(lambda: keep(cand, learned, 'again'), contains='uncommitted')
    subprocess.run(['git', '-C', str(learned), 'checkout', '-q', '--', '.'], check=True)
    test_eq(sorted(_files(learned)), ['AGENTS.md', 'skills/sizes/SKILL.md'])
    assert not (learned/'.kosha').exists()
    log = subprocess.run(['git', '-C', str(learned), 'log', '-1', '--format=%an %s'], capture_output=True, text=True).stdout.strip()
    test_eq(log, 'Dhrona Read sizes from ls')

## Evidence for the editor

The editor reads files, not a long prompt. `trajectory_md` writes one turn as its request, each tool call with its arguments, outcome and clipped result, and its reply. `findings_md` lists each run's route cost, Dhrona's findings and its failed or refused calls. `MISSION` tells the editor where the evidence and the candidate folder are, what it may write, and that changing nothing is a valid answer.

In [ ]:
#| export
def trajectory_md(turn, mx=1500):
    "One turn as markdown: the request, each tool call with its arguments, outcome and clipped result, then the reply."
    rows = [f"## Request\n\n{turn.get('prompt', '')}", '## Tool calls']
    for i, a in enumerate(turn.get('activity') or []):
        out = str(a.get('detail') or '')[:mx].replace('\n', '\n    ')
        rows.append(f"{i}. `{a.get('tool')}` {json.dumps(tool_args(a), default=str)} -> {'ok' if a.get('ok') else 'failed'}\n\n    {out}")
    rows.append(f"## Reply\n\n{turn.get('reply') or '(no reply)'}")
    return '\n\n'.join(rows)

def findings_md(turns):
    "Each run's route cost, Dhrona's findings and its failed or refused calls, as markdown."
    rows = []
    for n, t in enumerate(turns, 1):
        acts = t.get('activity') or []
        rows.append(f'## base-{n}: route cost {route_cost(t)} (findings, failed calls, calls)')
        rows += [f'- finding at call {f.index}: {f.message}' for f in assess_turn(t).findings]
        rows += [f"- call {i} `{a.get('tool')}` failed: {str(a.get('detail') or '')[:300]}" for i, a in enumerate(acts) if a.get('ok') is False]
    return '\n'.join(rows)

MISSION = r"""You improve how a coding agent, Ramabana, works by editing its learned folder, `{candidate}`.

Ramabana puts `AGENTS.md` from that folder into every briefing, after its own rules and before the project's instructions. It offers each `skills/<name>/SKILL.md` there as an on-demand skill, listed by the `name` and `description` in its frontmatter. Nothing else in the folder is read.

The evidence is in `{evidence}`. `task.md` is a request a person made. Each `base-N.md` is one run of that request under the folder as it stands. `findings.md` gives each run's route cost and what Dhrona's route check flagged. Read every evidence file and the current folder before you decide.

Look for a behaviour that repeats across the runs and costs the route something: a call that was refused or failed, a shell command where a dedicated tool does the job, or a call that repeats work already done. A defect in one run only is noise.

Change only what the evidence shows needs changing.
- Prefer a skill for a procedure with a recognisable trigger. It costs nothing until it is used.
- Use `AGENTS.md` only for a short rule that applies to every task. Keep the whole file under {mx} characters, and cut something to pay for what you add.
- Write nothing specific to this request or this repository. The folder applies to every project.
- Never change what the agent delivers, only how it works.

If the evidence argues for no change, change nothing. That is a valid answer.

Edit the files in `{candidate}` with your tools. Then reply with one line per change naming the run and tool call that motivated it, or with the single word KEEP if you changed nothing."""

In [ ]:
print(findings_md([base]))
print(trajectory_md(base)[:400])

## A round

`improve_round` takes a request and the `start` record Ramabana wrote for its session (`<cfg>/runs/<session>/start.json`: `repo`, `head`, `dirty`). A start with uncommitted changes raises `ValueError`, because a replay from `head` would not see them. So does a learned folder holding files outside its layout, before any run starts. `work` must not exist yet. Every run, the evidence, the candidate and `report.json` stay in it. An editor or candidate run that records no turn ends the round with the error in the report. Replays refuse every write by default; `approve='edits'` lets them edit files inside each run's throwaway clone, which shares no object files with the source repository. Any other `approve` raises `ValueError`, so a replay never runs a shell command. With `learned` set, a kept candidate is committed there. Without it, the round only reports.

In [ ]:
#| export
def _summary(turn):
    "What a report keeps of one turn."
    return dict(route=route_cost(turn), findings=[f.message for f in assess_turn(turn).findings],
                calls=[a.get('tool') for a in turn.get('activity') or []], reply=turn.get('reply'),
                cost=(turn.get('usage') or {}).get('cost'))

def improve_round(
    prompt,         # the request to replay
    start,          # `{repo, head, dirty}` from the session's `start.json`
    work,           # a new folder for this round's runs, evidence and report
    learned=None,   # the learned folder to start from and commit a kept candidate to
    model=None,     # the model the runs use
    editor=None,    # the model that edits the candidate
    k=2,            # base runs
    approve='ask',  # `edits` lets a replay edit files inside its throwaway clone
    exe=RAMABANA,
):
    "One round: `k` base runs, one edited learned folder, one run under it, and the decision; returns the report."
    if k < 1: raise ValueError('a round needs at least one base run')
    if approve not in ('ask', 'edits'): raise ValueError(f"approve is 'ask' or 'edits', not {approve!r}: a replay never runs shell commands")
    if start.get('dirty'): raise ValueError(f"{start['repo']} had uncommitted changes when the session began; a replay from {start['head']} would not see them")
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; a round writes into a new folder')
    current, cand, ev = work/'current', work/'candidate', work/'evidence'
    if learned is not None and Path(learned).exists(): shutil.copytree(learned, current, ignore=_HIDDEN)
    else: current.mkdir(parents=True)
    if (bad := check_learned(current)): raise ValueError(f'{learned} holds files a learned folder may not: {bad}')
    def run(name, folder): return run_turn(prompt, [replay_clone(start, work/name/'repo')], work/name/'cfg', model, folder, approve, exe=exe)
    bases = [run(f'base-{n}', current) for n in range(1, k+1)]
    ev.mkdir()
    (ev/'task.md').write_text(prompt)
    for n, b in enumerate(bases, 1): (ev/f'base-{n}.md').write_text(trajectory_md(b))
    (ev/'findings.md').write_text(findings_md(bases))
    shutil.copytree(current, cand, ignore=_HIDDEN)
    try: edit = run_turn(MISSION.format(candidate=cand, evidence=ev, mx=LEARNED_MAX), [cand, ev], work/'editor'/'cfg', editor,
                         approve='edits', exe=exe)
    except RuntimeError as e: edit = dict(error=str(e))
    report = dict(prompt=prompt, start=start, bases=[_summary(b) for b in bases], notes=edit.get('reply'), editor_error=edit.get('error'),
                  editor_cost=(edit.get('usage') or {}).get('cost'), changed=_files(cand) != _files(current),
                  problems=check_learned(cand), adopted=False)
    if report['changed'] and not report['problems'] and not report['editor_error']:
        try: c = run('candidate-run', cand)
        except RuntimeError as e: c = None; report['candidate_error'] = str(e)
        if c is not None:
            report['candidate'] = _summary(c)
            report['adopted'] = better(c, bases)
            if report['adopted'] and learned is not None:
                routes = ', '.join(str(b['route']) for b in report['bases'])
                msg = f"Lesson from a replay: {' '.join(prompt.split())[:60]}\n\n{report['notes']}\n\nRoute cost (findings, failed calls, calls): base {routes}; candidate {report['candidate']['route']}."
                report['commit'] = keep(cand, learned, msg)
    (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
    return report

In [ ]:
FAKE = r'''#!/usr/bin/env python3
"A stand-in for `ramabana`: the editor writes AGENTS.md; a run takes one call once AGENTS.md exists, three before."
import json, sys
from pathlib import Path
a = sys.argv[1:]
roots, cfg, prompt = a[a.index('--root') + 1].split(','), Path(a[a.index('--cfg') + 1]), a[-1]
ls = dict(tool='ls', ok=True, args={'path': ''})
if prompt.startswith('You improve'):
    Path(roots[0], 'AGENTS.md').write_text('Read file sizes from ls.\n')
    acts, reply = [dict(tool='create_file', ok=True, args={})], 'base-1 call 1: wrote AGENTS.md'
elif (cfg/'learned'/'AGENTS.md').exists(): acts, reply = [ls], 'big.py 50'
else: acts, reply = [ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls], 'big.py 50'
(cfg/'agent-history.jsonl').write_text(json.dumps(dict(prompt=prompt, activity=acts, reply=reply, usage={'cost': 0})) + '\n')
'''
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo = d/'fake-ramabana', d/'repo'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    start = dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=[])
    r = improve_round('sizes of the big files', start, d/'work', learned=d/'learned', exe=str(exe))
    test_eq((r['bases'][0]['route'], r['candidate']['route'], r['adopted']), ((0, 1, 3), (0, 0, 1), True))
    test_eq(sorted(_files(d/'learned')), ['AGENTS.md'])
    test_eq(sorted(_files(d/'work'/'candidate')), ['AGENTS.md'])   # the runs sit beside the candidate, never in it
    assert (d/'work'/'report.json').exists()
    test_fail(lambda: improve_round('x', start, d/'work', exe=str(exe)), contains='exists')
    test_fail(lambda: improve_round('x', start | dict(dirty=['big.py']), d/'w2', exe=str(exe)), contains='uncommitted')
    test_fail(lambda: improve_round('x', start, d/'w3', approve='auto', exe=str(exe)), contains='approve')

## Training on past sessions

A session that predates `start.json` records no starting commit, and most record no folder. `session_start` places a turn in the repository most of its absolute tool paths fall in, else in the first git repository among `roots`. Its `head` is that repository's last commit before the turn's `at`, an epoch in seconds or an ISO date; an unreadable `at` leaves the turn unplaced. `dirty` is None because uncommitted changes at that time are unknown. `trainable` keeps the first turn of each session whose request is self-contained, whose route cost something, and which made at most `max_calls` calls, shortest first. It drops a request that carried attachments, because the attached file belonged to that session. It drops a turn that committed, because `at` marks the turn's end and its start would hold that commit. A request asked in several sessions is kept once, from the cheapest. `trainable_sessions` reads Ramabana's own history and session index. It takes a session's start from the `start.json` entry for the repository the work used, and drops the session when that entry had uncommitted changes.

In [ ]:
#| export
def request_text(prompt):
    "The person's request inside `<user-request>` tags, else the whole prompt."
    found = re.findall(r'<user-request>\n?(.*?)\n?</user-request>', str(prompt or ''), re.S)
    return (found[-1] if found else str(prompt or '')).strip()

def _toplevel(path):
    "The git repository holding the absolute `path`, after `~` expansion, or None."
    p = Path(path).expanduser()
    if not p.is_absolute(): return None
    return next((str(d) for d in [p, *p.parents] if (d/'.git').exists()), None)

def _when(at):
    "A history row's `at`, an epoch in seconds or an ISO date, as an ISO date git understands; None when unreadable."
    try: return datetime.fromtimestamp(float(at), timezone.utc).isoformat()
    except (TypeError, ValueError, OverflowError, OSError): pass
    try: return datetime.fromisoformat(str(at)).isoformat()
    except ValueError: return None

def session_start(turn, roots=()):
    "`{repo, head, dirty, inferred}` for replaying `turn`'s session, or None when nothing places it in a repository."
    hits = Counter()
    for a in turn.get('activity') or []:
        for k in ('path', 'cwd', 'file', 'root'):
            if (r := _toplevel(str(tool_args(a).get(k) or ''))): hits[r] += 1
    repo = hits.most_common(1)[0][0] if hits else next((r for r in map(_toplevel, roots) if r), None)
    if repo is None or (when := _when(turn.get('at'))) is None: return None
    head = subprocess.run(['git', '-C', repo, 'rev-list', '-1', f'--before={when}', 'HEAD'], capture_output=True, text=True).stdout.strip()
    return dict(repo=repo, head=head, dirty=None, inferred=True) if head else None

def trainable(turns, roots_of=None, max_calls=40):
    "The first turn of each session that can seed a round, shortest first: `{session, prompt, model, start, route}`."
    first = {}
    for t in turns:
        if t.get('session'): first.setdefault(t['session'], t)
    found = []
    for sid, t in first.items():
        req, cost = request_text(t.get('prompt')), route_cost(t)
        if len(req) < 40 or '<attachments>' in str(t.get('prompt')) or t.get('error'): continue   # an attachment is a file only that session had
        if not 0 < cost[2] <= max_calls or cost[:2] == (0, 0): continue
        if any(a.get('tool') == 'git_commit' for a in t.get('activity') or []): continue    # `at` is the turn's end, so its start would hold this commit
        if (start := session_start(t, (roots_of or {}).get(sid) or ())) is None: continue
        found.append(dict(session=sid, prompt=req, model=t.get('model'), start=start, route=cost))
    out, seen = [], set()
    for s in sorted(found, key=lambda s: s['route'][2]):
        if s['prompt'] not in seen: seen.add(s['prompt']); out.append(s)
    return out

def trainable_sessions(cfg='~/.config/ramabana', max_calls=40):
    "`trainable` over Ramabana's history and session index in `cfg`, with each session's own `start.json` where it has a clean one."
    cfg = Path(cfg).expanduser()
    idx = cfg/'agent-sessions.json'
    roots = {s: v.get('roots') for s, v in json.loads(idx.read_text()).get('sessions', {}).items()} if idx.exists() else {}
    out = []
    for s in trainable(read_history(cfg/'agent-history.jsonl'), roots, max_calls):
        p = cfg/'runs'/s['session']/'start.json'
        rec = next((r for r in (json.loads(p.read_text()) if p.exists() else []) if r.get('repo') == s['start']['repo']), None)
        if rec is not None:
            if rec.get('dirty'): continue
            s['start'] = rec | dict(inferred=False)
        out.append(s)
    return out

In [ ]:
import os
from datetime import datetime, timezone
with TemporaryDirectory() as d:
    repo = Path(d)/'repo'; repo.mkdir()
    def git(*a, when=None):
        env = os.environ | ({'GIT_AUTHOR_DATE': when, 'GIT_COMMITTER_DATE': when} if when else {})
        return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True, env=env).stdout.strip()
    git('init', '-q')
    for n, when in [(1, '2026-01-01T00:00:00Z'), (2, '2026-03-01T00:00:00Z')]:
        (repo/'a.py').write_text(f'x = {n}\n'); git('add', '.')
        git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', f'c{n}', when=when)
    january = git('rev-list', '--max-parents=0', 'HEAD')
    t = {'session': 's1', 'at': datetime(2026, 2, 1, tzinfo=timezone.utc).timestamp(), 'model': 'claude-opus-5-5',
         'prompt': '<user-request>\nwhere is x defined, and which files read it? list them\n</user-request>',
         'activity': [dict(tool='grep', ok=True, args={'pattern': 'x', 'path': str(repo/'a.py')}),
                      dict(tool='run_shell', ok=False, args={'command': 'cat a.py'})]}
    test_eq((session_start(t)['repo'], session_start(t)['head']), (str(repo), january))   # the last commit before the turn
    test_eq(session_start(t | {'at': '2026-02-01T00:00:00+00:00'})['head'], january)        # an ISO `at` reads the same
    test_eq((session_start(t | {'activity': []}), session_start(t | {'at': None}), _toplevel('relative/dir')), (None, None, None))
    test_eq(session_start(t | {'activity': []}, roots=[str(repo)])['repo'], str(repo))
    attached = t | {'session': 's3', 'prompt': '<user-request>\nwhat does this screenshot show about the layout?\n</user-request>\n<attachments>\nimage /tmp/paste.png\n</attachments>'}
    cheaper = t | {'session': 's5', 'activity': [dict(tool='grep', ok=False, args={'pattern': 'x', 'path': str(repo/'a.py')})]}
    committed = t | {'session': 's6', 'prompt': 'where is x defined, and who imports it? list each file please',
                     'activity': t['activity'] + [dict(tool='git_commit', ok=True, args={'message': 'm'})]}
    got = trainable([t, t | {'prompt': 'a later turn in the same session'}, t | {'session': 's2', 'prompt': 'ok'},
                     t | {'session': 's4'}, attached, cheaper, committed])
    test_eq([(s['session'], s['prompt'], s['model']) for s in got], [('s5', 'where is x defined, and which files read it? list them', 'claude-opus-5-5')])
    # s4 and s5 ask what s1 asked and s5 costs least; s3 names a file only its session had; s6's start would hold its own commit
    cfg = Path(d)/'cfg'; (cfg/'runs'/'s1').mkdir(parents=True)
    (cfg/'agent-history.jsonl').write_text(json.dumps(t) + '\n')
    rec = [dict(repo='/elsewhere', head='abc', dirty=[]), dict(folder=str(repo), repo=str(repo), head=january, dirty=[])]
    (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq([(s['start']['head'], s['start']['inferred']) for s in trainable_sessions(cfg)], [(january, False)])   # the entry for the repository the work used
    rec[1]['dirty'] = ['a.py']; (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq(trainable_sessions(cfg), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()